In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd

# Load dataset
df = pd.read_csv('D:/PERKULIAHAN/Kuliah Semester 6/[DS] Pemrosesan Bahasa Alami/Tugas/Tugas 6/BERT Twitter Sentiment/dataset/twitter_sentiment.csv')

# Tokenisasi menggunakan BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenisasi dan padding menggunakan PyTorch
def tokenize_and_pad(texts):
    return tokenizer(texts.tolist(), padding=True, truncation=True, return_tensors="pt", max_length=128)

# Tokenisasi teks
tokens = tokenize_and_pad(df['text'])

# Label encoding untuk sentimen
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(df['sentiment'])

# Split data menjadi training dan test
X_train, X_test, y_train, y_test = train_test_split(tokens['input_ids'], labels, test_size=0.2, random_state=42)

# Padding secara manual dengan PyTorch 
def pad_sequences_pytorch(X, maxlen=128):
    return torch.cat([X, torch.zeros(X.shape[0], maxlen - X.shape[1]).long()], dim=1) if X.shape[1] < maxlen else X[:, :maxlen]

X_train_padded = pad_sequences_pytorch(X_train)
X_test_padded = pad_sequences_pytorch(X_test)

# Convert to PyTorch tensors with correct data type for labels
X_train_tensor = torch.tensor(X_train_padded)
X_test_tensor = torch.tensor(X_test_padded)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)  # Ensure long type
y_test_tensor = torch.tensor(y_test, dtype=torch.long)    # Ensure long type

# Create DataLoader untuk batch processing
train_data = TensorDataset(X_train_tensor, y_train_tensor)
test_data = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class CNN_Text(nn.Module):
    def __init__(self, vocab_size, embedding_dim, kernel_sizes, num_filters, num_classes):
        super(CNN_Text, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.convs = nn.ModuleList([ 
            nn.Conv2d(1, num_filters, (kernel_size, embedding_dim)) for kernel_size in kernel_sizes
        ])
        
        self.fc = nn.Linear(len(kernel_sizes) * num_filters, num_classes)

    def forward(self, x):
        x = self.embedding(x)  # (batch_size, seq_len) -> (batch_size, seq_len, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, seq_len, embedding_dim)

        conv_results = [torch.relu(conv(x)).squeeze(3) for conv in self.convs]  # Apply convolutions
        pooled_results = [torch.max(conv, dim=2)[0] for conv in conv_results]  # Max pooling
        x = torch.cat(pooled_results, dim=1)  # Concatenate results from each filter
        x = self.fc(x)
        
        return x

# Initialize model
vocab_size = len(tokenizer.vocab)  # Vocabulary size
embedding_dim = 100  # Embedding dimension
kernel_sizes = [3, 4, 5]  # Filter sizes
num_filters = 100  # Number of filters
num_classes = 2  # Sentiment classes (positive, negative)

model = CNN_Text(vocab_size, embedding_dim, kernel_sizes, num_filters, num_classes)

# Loss function dan optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
def train(model, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        
        # Calculate loss and backpropagate
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        # Track performance
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    print(f'Epoch {epoch}, Loss: {total_loss/len(train_loader)}, Accuracy: {correct/total}')

# Train model for several epochs
for epoch in range(1, 11):
    train(model, train_loader, criterion, optimizer, epoch)

# Evaluasi model
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    print(f'Test Accuracy: {accuracy}')
    
evaluate(model, test_loader)

# Fungsi untuk memproses input dan prediksi sentimen
def predict_sentiment(text, model, tokenizer):
    # Tokenisasi teks
    tokens = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=128)
    
    # Mengambil input_ids yang telah di-tokenize
    input_ids = tokens['input_ids']
    
    # Membuat tensor dari input
    input_tensor = torch.tensor(input_ids)

    # Memastikan model berada dalam mode evaluasi (non-training mode)
    model.eval()
    
    with torch.no_grad():
        # Melakukan prediksi
        output = model(input_tensor)
        
        # Mengambil indeks dengan nilai tertinggi untuk menentukan prediksi
        _, predicted = torch.max(output, 1)
        
    # Menyajikan hasil
    sentiment = 'Positive' if predicted.item() == 1 else 'Negative'
    return sentiment

# Mengambil input dari pengguna di terminal
input_text = input("Masukkan teks untuk prediksi sentimen: ")

# Melakukan prediksi sentimen
sentiment = predict_sentiment(input_text, model, tokenizer)

# Menampilkan hasil prediksi
print(f"Sentimen dari teks yang dimasukkan: {sentiment}")


C:\Users\USER\AppData\Local\Temp\ipykernel_4192\4206136503.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train_tensor = torch.tensor(X_train_padded)
C:\Users\USER\AppData\Local\Temp\ipykernel_4192\4206136503.py:39: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test_tensor = torch.tensor(X_test_padded)


Epoch 1, Loss: 0.5471290089130402, Accuracy: 0.70585
Epoch 2, Loss: 0.32620308084487915, Accuracy: 0.86355
Epoch 3, Loss: 0.15283163896799087, Accuracy: 0.95455
Epoch 4, Loss: 0.047437392973899845, Accuracy: 0.99435
Epoch 5, Loss: 0.011877560524642467, Accuracy: 1.0
Epoch 6, Loss: 0.004226826940476895, Accuracy: 1.0
Epoch 7, Loss: 0.002360876833088696, Accuracy: 1.0
Epoch 8, Loss: 0.0014949536042287946, Accuracy: 1.0
Epoch 9, Loss: 0.0009829950853716583, Accuracy: 1.0
Epoch 10, Loss: 0.0006624530605506152, Accuracy: 1.0
Test Accuracy: 0.8284
Sentimen dari teks yang dimasukkan: Positive


C:\Users\USER\AppData\Local\Temp\ipykernel_4192\4206136503.py:142: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_ids)
